# 📊 Retail Revenue Optimization — Exploratory Data Analysis

**Dataset:** [Retail Sales Dataset](https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset) (Mohammad Talib, CC0)
**Author:** Kigozi Ramathan
**Goal:** Diagnose declining monthly revenue and recommend actions.

---

> 💡 **Before running:** run `python scripts/download_data.py` once to fetch the dataset into `data/raw/`.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('Setup complete ✅')

## 2. Load Data

In [ ]:
from pathlib import Path

# Find the CSV file automatically
csv_files = list(Path('../data/raw').glob('*.csv'))
print('Found:', csv_files)

df = pd.read_csv(csv_files[0])
df.head()

In [ ]:
# Basic shape and info
print('Shape:', df.shape)
df.info()

## 3. Data Cleaning

In [ ]:
# 3.1 Missing values
df.isnull().sum()

In [ ]:
# 3.2 Duplicates
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
# 3.3 Fix data types — parse the Date column
df['Date'] = pd.to_datetime(df['Date'])

# 3.4 Add useful time features for trend analysis
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Day_of_Week'] = df['Date'].dt.day_name()
df.head()

In [ ]:
# 3.5 Sanity check: Total Amount should equal Quantity * Price per Unit
df['Calc_Total'] = df['Quantity'] * df['Price per Unit']
mismatch = (df['Total Amount'] - df['Calc_Total']).abs() > 0.01
print('Rows where Total != Quantity * Price:', mismatch.sum())
df = df.drop(columns=['Calc_Total'])

## 4. Exploratory Data Analysis

### 4.1 Revenue trend over time

In [ ]:
monthly_revenue = df.groupby(df['Date'].dt.to_period('M'))['Total Amount'].sum()

plt.figure(figsize=(12, 5))
monthly_revenue.plot(marker='o', color='#1f77b4')
plt.title('Total Monthly Revenue')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.2 Revenue by product category

In [ ]:
cat_rev = df.groupby('Product Category')['Total Amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=cat_rev.values, y=cat_rev.index, palette='viridis')
plt.title('Revenue by Product Category')
plt.xlabel('Total Revenue')
plt.tight_layout()
plt.show()

### 4.3 Customer demographics: age & gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0])
axes[0].set_title('Customer Age Distribution')

gender_rev = df.groupby('Gender')['Total Amount'].sum()
sns.barplot(x=gender_rev.index, y=gender_rev.values, ax=axes[1])
axes[1].set_title('Revenue by Gender')

plt.tight_layout()
plt.show()

### 4.4 Top categories vs. quantity sold

In [ ]:
cat_qty = df.groupby('Product Category')['Quantity'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=cat_qty.values, y=cat_qty.index, palette='magma')
plt.title('Units Sold by Product Category')
plt.xlabel('Total Quantity')
plt.tight_layout()
plt.show()

## 5. Insights & Recommendations

Summarize your findings here, e.g.:
- Which month shows the sharpest revenue decline?
- Which product category is underperforming?
- Which customer segment should the business target?
- What concrete actions reverse the trend?